# 🎙️ VITS Fine-tuning Notebook
**Model:** `male_vits_23_dec_2024` | **Language:** Bengali (bn) | **Phonemizer:** espeak

This notebook fine-tunes a pre-trained VITS model using the Coqui TTS trainer.

---
### Pipeline Overview
1. Environment setup & dependency check
2. Config generation with fine-tune parameters
3. Dataset validation
4. Phoneme cache preparation
5. Training
6. Evaluation & inference test

## 📦 1. Install & Import Dependencies

In [ ]:
# Install Coqui TTS if not already installed
!pip install TTS --quiet
!pip install tensorboard --quiet

# Verify espeak is available (required for Bengali phonemization)
!which espeak || apt-get install -y espeak espeak-data

In [ ]:
import os
import json
import shutil
import torch
import pandas as pd
from pathlib import Path
from IPython.display import Audio, display

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## ⚙️ 2. Configuration

In [ ]:
# ============================================================
# 🔧 EDIT THESE PATHS BEFORE RUNNING
# ============================================================

BASE_DIR          = "/teamspace/studios/this_studio"
ORIGINAL_CKPT     = f"{BASE_DIR}/model/male_vits_23_dec_2024-December-23-2024_09+14AM-0000000/checkpoint_140000.pth"
FINETUNE_DATA_DIR = f"{BASE_DIR}/data/data/"                     # path to your fine-tune audio
META_FILE_TRAIN   = f"{BASE_DIR}/data/data/metadata.csv"         # pipe-separated: wav_path|text
META_FILE_VAL     = ""                                            # leave empty to auto-split
OUTPUT_DIR        = f"{BASE_DIR}/model/finetune_output"
PHONEME_CACHE_DIR = f"{OUTPUT_DIR}/phoneme_cache"
CONFIG_SAVE_PATH  = f"{OUTPUT_DIR}/finetune_config.json"

# ============================================================
# 🎛️ FINE-TUNE HYPERPARAMETERS
# ============================================================
RUN_NAME        = "male_vits_finetune_v1"
EPOCHS          = 200          # Lower than original (was 1000)
BATCH_SIZE      = 16           # Reduce if OOM (was 48)
EVAL_BATCH_SIZE = 16
LR_GEN          = 2e-5         # 10x lower than original (was 2e-4)
LR_DISC         = 2e-5
LR_GAMMA        = 0.9999       # Slightly faster decay for fine-tuning
SAVE_STEP       = 1000         # More frequent saves (was 5000)
PRINT_STEP      = 100
PLOT_STEP       = 50
MEL_LOSS_ALPHA  = 80.0         # Increased from 45 for better quality
GRAD_CLIP       = 5.0          # Tighter than original 1000.0

# Freeze layers? Set True to freeze when adapting to new speaker
FREEZE_ENCODER          = False
FREEZE_POSTERIOR_ENC    = False  # Set True if you have < 30 min data
FREEZE_DURATION_PRED    = False
FREEZE_FLOW_DECODER     = False
FREEZE_WAVEFORM_DECODER = False

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PHONEME_CACHE_DIR, exist_ok=True)
print("✅ Directories created")

In [ ]:
# Build fine-tune config (derived from original config.json)
finetune_config = {
    # --- Run metadata ---
    "output_path": OUTPUT_DIR,
    "run_name": RUN_NAME,
    "run_description": "VITS Bengali fine-tune from checkpoint_140000",
    "logger_uri": None,
    "project_name": None,
    "dashboard_logger": "tensorboard",

    # --- Logging & checkpointing ---
    "print_step": PRINT_STEP,
    "plot_step": PLOT_STEP,
    "save_step": SAVE_STEP,
    "save_n_checkpoints": 5,
    "save_checkpoints": True,
    "save_all_best": True,       # Save every best checkpoint during fine-tune
    "save_best_after": 500,
    "save_on_interrupt": True,
    "print_eval": True,
    "model_param_stats": False,
    "log_model_step": None,

    # --- Training control ---
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "grad_clip": [GRAD_CLIP, GRAD_CLIP],
    "scheduler_after_epoch": True,
    "mixed_precision": False,
    "precision": "fp16",
    "use_grad_scaler": False,
    "training_seed": 54321,

    # --- Optimizer ---
    "lr": 0.0002,               # fallback lr (not used directly by VITS)
    "optimizer": "AdamW",
    "optimizer_params": {
        "betas": [0.8, 0.99],
        "eps": 1e-9,
        "weight_decay": 0.01
    },
    "lr_gen": LR_GEN,
    "lr_disc": LR_DISC,
    "lr_scheduler_gen": "ExponentialLR",
    "lr_scheduler_gen_params": {"gamma": LR_GAMMA, "last_epoch": -1},
    "lr_scheduler_disc": "ExponentialLR",
    "lr_scheduler_disc_params": {"gamma": LR_GAMMA, "last_epoch": -1},

    # --- Loss weights ---
    "kl_loss_alpha": 1.0,
    "disc_loss_alpha": 1.0,
    "gen_loss_alpha": 1.0,
    "feat_loss_alpha": 1.0,
    "mel_loss_alpha": MEL_LOSS_ALPHA,
    "dur_loss_alpha": 1.0,
    "speaker_encoder_loss_alpha": 1.0,

    # --- Hardware ---
    "distributed_backend": "nccl",
    "distributed_url": "tcp://localhost:54321",
    "allow_tf32": False,
    "cudnn_enable": True,
    "cudnn_deterministic": False,
    "cudnn_benchmark": True,

    # --- Model ---
    "model": "vits",
    "num_loader_workers": 8,
    "num_eval_loader_workers": 4,
    "use_noise_augment": False,
    "return_wav": True,

    # --- Audio ---
    "audio": {
        "fft_size": 1024,
        "sample_rate": 22050,
        "win_length": 1024,
        "hop_length": 256,
        "num_mels": 80,
        "mel_fmin": 0,
        "mel_fmax": None
    },

    # --- Text / Phonemes ---
    "use_phonemes": True,
    "phonemizer": "espeak",
    "phoneme_language": "bn",
    "compute_input_seq_cache": True,
    "text_cleaner": "phoneme_cleaners",
    "enable_eos_bos_chars": False,
    "test_sentences_file": "",
    "phoneme_cache_path": PHONEME_CACHE_DIR,
    "add_blank": True,

    "characters": {
        "characters_class": "TTS.tts.utils.text.characters.IPAPhonemes",
        "vocab_dict": None,
        "pad": "<PAD>", "eos": "<EOS>", "bos": "<BOS>", "blank": "<BLNK>",
        "characters": "iyɨʉɯuɪʏʊeøɘəɵɤoɛœɜɞʌɔæɐaɶɑɒᵻʘɓǀɗǃʄǂɠǁʡpbtdʈɖcɟkɡqɢʔɴŋɲɳnɱmʙrʀⱱɾɽɸβfvθðsząʃʒʂʐçʝxɣχʁħʕhɦɬɮʋɹɻjɰlɭʎʟˈˌːˑʍwɥʜʢʡɕʑɺɧʲɚ˞ɫ",
        "punctuations": "!'(),-.:;? ",
        "phonemes": None,
        "is_unique": False,
        "is_sorted": True
    },

    # --- Data loading ---
    "batch_group_size": 0,
    "loss_masking": None,
    "min_audio_len": 1,
    "max_audio_len": None,
    "min_text_len": 1,
    "max_text_len": None,
    "compute_f0": False,
    "compute_energy": False,
    "compute_linear_spec": True,
    "precompute_num_workers": 4,
    "start_by_longest": False,
    "shuffle": True,
    "drop_last": True,
    "eval_split_max_size": None,
    "eval_split_size": 0.05,    # 5% eval (larger than original 1%)

    # --- Dataset ---
    "datasets": [{
        "formatter": "",
        "dataset_name": "",
        "path": FINETUNE_DATA_DIR,
        "meta_file_train": META_FILE_TRAIN,
        "ignored_speakers": None,
        "language": "",
        "phonemizer": "",
        "meta_file_val": META_FILE_VAL,
        "meta_file_attn_mask": ""
    }],

    # --- Sampler ---
    "use_speaker_weighted_sampler": False,
    "speaker_weighted_sampler_alpha": 1.0,
    "use_language_weighted_sampler": True,
    "language_weighted_sampler_alpha": 1.0,
    "use_length_weighted_sampler": False,
    "length_weighted_sampler_alpha": 1.0,
    "use_weighted_sampler": False,
    "weighted_sampler_attrs": None,
    "weighted_sampler_multipliers": None,

    # --- Test sentences (Bengali) ---
    "test_sentences": [
        "আমার   সোনার বাংলা, আমি তোমায় ভালোবাসি।",
        "চিরদিন   তোমার আকাশ, তোমার বাতাস, আমার প্রাণে বাজায় বাঁশি",
        "ও মা,   ফাগুনে তোর আমের বনে ঘ্রাণে পাগল করে,মরি হায়, হায় রে।"
    ],
    "run_eval": True,
    "run_eval_steps": None,
    "test_delay_epochs": 1,
    "target_loss": None,

    # --- Model args (same as base model) ---
    "model_args": {
        "num_chars": 131,
        "out_channels": 513,
        "spec_segment_size": 32,
        "hidden_channels": 192,
        "hidden_channels_ffn_text_encoder": 768,
        "num_heads_text_encoder": 2,
        "num_layers_text_encoder": 6,
        "kernel_size_text_encoder": 3,
        "dropout_p_text_encoder": 0.1,
        "dropout_p_duration_predictor": 0.5,
        "kernel_size_posterior_encoder": 5,
        "dilation_rate_posterior_encoder": 1,
        "num_layers_posterior_encoder": 16,
        "kernel_size_flow": 5,
        "dilation_rate_flow": 1,
        "num_layers_flow": 4,
        "resblock_type_decoder": "1",
        "resblock_kernel_sizes_decoder": [3, 7, 11],
        "resblock_dilation_sizes_decoder": [[1,3,5],[1,3,5],[1,3,5]],
        "upsample_rates_decoder": [8, 8, 2, 2],
        "upsample_initial_channel_decoder": 512,
        "upsample_kernel_sizes_decoder": [16, 16, 4, 4],
        "periods_multi_period_discriminator": [2, 3, 5, 7, 11],
        "use_sdp": True,
        "noise_scale": 1.0,
        "inference_noise_scale": 0.5,    # Reduced from 0.667 → cleaner output
        "length_scale": 1.0,
        "noise_scale_dp": 1.0,
        "inference_noise_scale_dp": 1.0,
        "max_inference_len": None,
        "init_discriminator": True,
        "use_spectral_norm_disriminator": False,
        "use_speaker_embedding": False,
        "num_speakers": 0,
        "speakers_file": None,
        "d_vector_file": None,
        "speaker_embedding_channels": 256,
        "use_d_vector_file": False,
        "d_vector_dim": 0,
        "detach_dp_input": True,
        "use_language_embedding": False,
        "embedded_language_dim": 4,
        "num_languages": 0,
        "language_ids_file": None,
        "use_speaker_encoder_as_loss": False,
        "speaker_encoder_config_path": "",
        "speaker_encoder_model_path": "",
        "condition_dp_on_speaker": True,
        # --- Layer freezing ---
        "freeze_encoder": FREEZE_ENCODER,
        "freeze_DP": FREEZE_DURATION_PRED,
        "freeze_PE": FREEZE_POSTERIOR_ENC,
        "freeze_flow_decoder": FREEZE_FLOW_DECODER,
        "freeze_waveform_decoder": FREEZE_WAVEFORM_DECODER,
        "encoder_sample_rate": None,
        "interpolate_z": True,
        "reinit_DP": False,
        "reinit_text_encoder": False
    },

    # --- Misc ---
    "r": 1,
    "num_speakers": 0,
    "use_speaker_embedding": False,
    "speakers_file": None,
    "speaker_embedding_channels": 256,
    "language_ids_file": None,
    "use_language_embedding": False,
    "use_d_vector_file": False,
    "d_vector_file": None,
    "d_vector_dim": 0,

    # --- Restore from checkpoint ---
    "restore_path": ORIGINAL_CKPT,
    "github_branch": "inside_docker"
}

with open(CONFIG_SAVE_PATH, "w") as f:
    json.dump(finetune_config, f, indent=4, ensure_ascii=False)

print(f"✅ Fine-tune config saved to: {CONFIG_SAVE_PATH}")

## 🔍 3. Validate Dataset

In [ ]:
import librosa

def validate_dataset(meta_path, audio_base_dir, sample_rate=22050, max_check=20):
    """
    Validates metadata CSV and audio files.
    Expected CSV format:  wav_filename|transcription
    """
    print(f"📂 Checking metadata: {meta_path}")
    assert os.path.exists(meta_path), f"❌ Metadata not found: {meta_path}"

    df = pd.read_csv(meta_path, sep="|", header=None, names=["wav", "text"])
    print(f"   Total samples   : {len(df)}")
    print(f"   Sample rows:")
    print(df.head(3).to_string(index=False))

    missing, bad_sr, durations = [], [], []
    for i, row in df.iterrows():
        if i >= max_check:
            print(f"   (Checked first {max_check} files — set max_check=len(df) for full scan)")
            break
        wav_path = os.path.join(audio_base_dir, "wavs", row["wav"] + ".wav")
        if not wav_path.endswith(".wav"):
            wav_path = os.path.join(audio_base_dir, row["wav"])
        if not os.path.exists(wav_path):
            missing.append(wav_path)
            continue
        try:
            y, sr = librosa.load(wav_path, sr=None)
            if sr != sample_rate:
                bad_sr.append((wav_path, sr))
            durations.append(len(y) / sr)
        except Exception as e:
            print(f"   ⚠️  Error loading {wav_path}: {e}")

    if missing:
        print(f"\n❌ Missing files ({len(missing)}): {missing[:5]}")
    if bad_sr:
        print(f"\n⚠️  Wrong sample rate files ({len(bad_sr)}): {bad_sr[:3]}")
        print("   → Resample to 22050 Hz before training")
    if durations:
        total_mins = sum(durations) / 60
        print(f"\n📊 Audio stats (first {len(durations)} files):")
        print(f"   Min duration : {min(durations):.2f}s")
        print(f"   Max duration : {max(durations):.2f}s")
        print(f"   Mean duration: {sum(durations)/len(durations):.2f}s")
        print(f"   Total (sample): {total_mins:.1f} min")
    if not missing and not bad_sr:
        print("\n✅ Dataset looks good!")

validate_dataset(META_FILE_TRAIN, FINETUNE_DATA_DIR)

## 🔄 4. Resample Audio (if needed)

In [ ]:
# Run this cell only if validate_dataset reported wrong sample rates
import soundfile as sf

def resample_dataset(audio_dir, target_sr=22050):
    wav_files = list(Path(audio_dir).rglob("*.wav"))
    print(f"Found {len(wav_files)} wav files. Resampling to {target_sr} Hz...")
    for wav_path in wav_files:
        y, sr = librosa.load(str(wav_path), sr=None)
        if sr != target_sr:
            y_resampled = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
            sf.write(str(wav_path), y_resampled, target_sr)
    print("✅ Resampling done.")

# Uncomment to run:
# resample_dataset(FINETUNE_DATA_DIR)

## 🗃️ 5. Warm-up Phoneme Cache

In [ ]:
# Clear old phoneme cache to avoid stale data conflicts
if os.path.exists(PHONEME_CACHE_DIR):
    shutil.rmtree(PHONEME_CACHE_DIR)
    os.makedirs(PHONEME_CACHE_DIR)
    print(f"🗑️  Cleared old phoneme cache")

# Test espeak Bengali phonemization
test_text = "আমার সোনার বাংলা"
result = os.popen(f'espeak -v bn --ipa -q "{test_text}"').read().strip()
print(f"Test phonemization:")
print(f"  Input : {test_text}")
print(f"  Output: {result}")
if result:
    print("✅ espeak Bengali phonemization working")
else:
    print("❌ espeak not working — install: apt-get install espeak espeak-data")

## ✅ 6. Pre-training Checks

In [ ]:
print("=" * 55)
print("PRE-TRAINING CHECKLIST")
print("=" * 55)

checks = {
    "Checkpoint exists"      : os.path.exists(ORIGINAL_CKPT),
    "Config saved"           : os.path.exists(CONFIG_SAVE_PATH),
    "Training metadata exists": os.path.exists(META_FILE_TRAIN),
    "Data directory exists"  : os.path.exists(FINETUNE_DATA_DIR),
    "CUDA available"         : torch.cuda.is_available(),
    "Output dir writable"    : os.access(OUTPUT_DIR, os.W_OK),
}

all_pass = True
for check, status in checks.items():
    icon = "✅" if status else "❌"
    print(f"  {icon}  {check}")
    if not status:
        all_pass = False

print("=" * 55)
if all_pass:
    print("🟢 All checks passed — ready to train!")
else:
    print("🔴 Fix failing checks before running training.")

## 🚀 7. Start Training

In [ ]:
# Launch TensorBoard in the background
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}

In [ ]:
# ▶️ Single GPU training
!python -m TTS.bin.train_tts \
    --config_path "{CONFIG_SAVE_PATH}"

In [ ]:
# ▶️ Multi-GPU training (uncomment if you have multiple GPUs)
# NUM_GPUS = torch.cuda.device_count()
# print(f"Launching on {NUM_GPUS} GPUs")
# !python -m torch.distributed.launch \
#     --nproc_per_node={NUM_GPUS} \
#     TTS/bin/train_tts.py \
#     --config_path "{CONFIG_SAVE_PATH}"

## 🔎 8. Find Best Checkpoint

In [ ]:
import glob

def find_best_checkpoint(output_dir):
    """Find the best checkpoint based on filename or modification time."""
    # Coqui TTS names best checkpoint 'best_model.pth'
    best = glob.glob(os.path.join(output_dir, "**", "best_model.pth"), recursive=True)
    if best:
        print(f"🏆 Best model: {best[0]}")
        return best[0]

    # Fallback: most recent checkpoint
    all_ckpts = sorted(
        glob.glob(os.path.join(output_dir, "**", "checkpoint_*.pth"), recursive=True),
        key=os.path.getmtime
    )
    if all_ckpts:
        latest = all_ckpts[-1]
        print(f"📌 Latest checkpoint: {latest}")
        print(f"   All checkpoints: {[os.path.basename(c) for c in all_ckpts]}")
        return latest

    print("❌ No checkpoints found yet.")
    return None

BEST_CKPT = find_best_checkpoint(OUTPUT_DIR)

## 🎧 9. Inference Test

In [ ]:
from TTS.api import TTS

# Load fine-tuned model
assert BEST_CKPT is not None, "No checkpoint found — train the model first!"

tts = TTS(model_path=BEST_CKPT, config_path=CONFIG_SAVE_PATH, progress_bar=False)

# Test sentences
test_sentences = [
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "চিরদিন তোমার আকাশ, তোমার বাতাস, আমার প্রাণে বাজায় বাঁশি",
    "ও মা, ফাগুনে তোর আমের বনে ঘ্রাণে পাগল করে।"
]

output_wavs = []
for i, sentence in enumerate(test_sentences):
    out_path = os.path.join(OUTPUT_DIR, f"test_output_{i+1}.wav")
    tts.tts_to_file(text=sentence, file_path=out_path)
    output_wavs.append(out_path)
    print(f"\n🔊 Sentence {i+1}: {sentence}")
    display(Audio(out_path))

print("\n✅ Inference test complete!")

## 📊 10. Compare: Base vs Fine-tuned

In [ ]:
# Side-by-side audio comparison
tts_base = TTS(model_path=ORIGINAL_CKPT, config_path=CONFIG_SAVE_PATH, progress_bar=False)
tts_ft   = TTS(model_path=BEST_CKPT,    config_path=CONFIG_SAVE_PATH, progress_bar=False)

compare_text = "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।"

base_out = os.path.join(OUTPUT_DIR, "compare_base.wav")
ft_out   = os.path.join(OUTPUT_DIR, "compare_finetuned.wav")

tts_base.tts_to_file(text=compare_text, file_path=base_out)
tts_ft.tts_to_file(text=compare_text,   file_path=ft_out)

print("🔷 Base model output:")
display(Audio(base_out))
print("🟢 Fine-tuned model output:")
display(Audio(ft_out))

## 💾 11. Export Final Model

In [ ]:
EXPORT_DIR = os.path.join(OUTPUT_DIR, "export")
os.makedirs(EXPORT_DIR, exist_ok=True)

# Copy best checkpoint + config to export directory
if BEST_CKPT:
    shutil.copy(BEST_CKPT, os.path.join(EXPORT_DIR, "model.pth"))
    shutil.copy(CONFIG_SAVE_PATH, os.path.join(EXPORT_DIR, "config.json"))
    print(f"✅ Model exported to: {EXPORT_DIR}")
    print(f"   Files:")
    for f in os.listdir(EXPORT_DIR):
        size = os.path.getsize(os.path.join(EXPORT_DIR, f)) / 1e6
        print(f"     {f}  ({size:.1f} MB)")
else:
    print("❌ No checkpoint to export.")

---
## 📋 Quick Reference

| Parameter | Base model | Fine-tune |
|---|---|---|
| `lr_gen / lr_disc` | 2e-4 | **2e-5** |
| `batch_size` | 48 | **16** |
| `epochs` | 1000 | **200** |
| `mel_loss_alpha` | 45 | **80** |
| `grad_clip` | 1000 | **5** |
| `inference_noise_scale` | 0.667 | **0.5** |
| `save_step` | 5000 | **1000** |
| `eval_split_size` | 1% | **5%** |

### 🛠 Troubleshooting
- **OOM error** → Reduce `batch_size` to 8 or 4
- **Loss diverges** → Lower `lr_gen/lr_disc` by 5x more, set `grad_clip` to 1.0
- **Robotic output** → Increase `mel_loss_alpha` to 100, train longer
- **Muffled audio** → Try `inference_noise_scale = 0.667` (back to original)
- **Phoneme errors** → Check espeak-ng version: `espeak-ng --version`